# Fundamentals 11 - Single AgenticSystem End-to-End

Objetivo: integrar en un solo recorrido los fundamentos de System (08) y Environment/Eval (10) sin introducir composicion multiagente.

El `AgenticSystem` es dueno del agente determinista y del explainer LM opcional. La respuesta de negocio no depende del LM: si OpenAI, Bedrock o vLLM no estan disponibles, la ejecución principal y la evaluacion siguen siendo validas.

```text
Input -> AgenticSystem -> Agent -> RunResult -> Lineage
                                  -> Environment -> Eval
                                  -> optional LM explanation
```


In [ ]:
import os
import agentic_systems as toolkit
PRETTY = False
scheduler = toolkit.scheduler(timeout_s=60, max_retries=0, max_tool_calls=6, max_turns=6)
local_runtime = toolkit.runtime(provider="python-runtime", model="python-runtime", region="local", scheduler=scheduler)
lm_runtime = toolkit.runtime(provider="auto", scheduler=scheduler)
lm_resolution = lm_runtime.describe()
force_local_only = bool(os.getenv("AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS"))
lm_available = lm_resolution["selected_provider"] != "auto" and not force_local_only
USER_PROMPT = "Empieza con 10, suma 20, resta 9, multiplica por 4 y divide entre 2."
NUMBERS = [10, 20, 9, 4, 2]
EXPECTED = 42
toolkit.show({
    "local_runtime": local_runtime.describe(),
    "lm_runtime": lm_resolution,
    "lm_available": lm_available,
    "force_local_only": force_local_only,
}, title="Runtime selection")


## Escenario didactico compartido

Se conserva el mismo problema aritmetico de los notebooks anteriores. Eso permite comparar la arquitectura sin cambiar el dominio ni el resultado esperado.


In [ ]:
toolkit.show({
    "prompt": USER_PROMPT,
    "numbers": NUMBERS,
    "expected": EXPECTED,
}, title="Shared arithmetic scenario")

## Parámetros de la integración single-agent

| Parametro | Qué controla | Decision del notebook |
|---|---|---|
| `local_runtime` | Ruta obligatoria de negocio. | `python-runtime` para una base determinista. |
| `lm_runtime` | Provider del explainer opcional. | `provider="auto"` para OpenAI, Bedrock o vLLM. |
| `AGENTIC_SYSTEMS_PROVIDER_PRIORITY` | Orden de seleccion de Providers. | Permite cambiar backend sin editar el notebook. |
| `AGENTIC_SYSTEMS_FORCE_LOCAL_TUTORIALS` | Desactiva llamadas LM. | Util para CI y pruebas sin credenciales. |
| `max_tool_calls=1` | Limite del agente principal. | El contrato exige exactamente `solve_arithmetic`. |
| `temperature=0.0` | Variabilidad del agente. | Mantiene la reproducibilidad del tutorial. |
| `trace="compact"` | Nivel de evidencia. | Conserva lineage legible sin ruido de debug. |
| `EXPECTED=42` | Criterio de Environment y Eval. | La evaluacion usa el mismo resultado esperado. |


## 1) Declarar la Tool del caso

La Tool contiene la operacion de negocio. El Agent y el Environment reutilizan la misma capacidad; no mantienen dos implementaciones del calculo.


In [ ]:
@toolkit.tool
def solve_arithmetic(numbers: list[int]) -> dict:
    a,b,c,d,e = numbers
    x1 = a + b; x2 = x1 - c; x3 = x2 * d; x4 = x3 / e
    return {"procedure": [f"{a}+{b}={x1}", f"{x1}-{c}={x2}", f"{x2}*{d}={x3}", f"{x3}/{e}={x4:g}"], "result": int(x4) if float(x4).is_integer() else x4}


## 2) Crear el System, el contrato y sus Agents

El agente `single_solver` es obligatorio y determinista. `single_lm_explainer` pertenece al mismo System, pero su contribucion es opcional y puede degradarse explicitamente.


In [ ]:
@toolkit.tool
def record_review(summary: str) -> dict:
    """Registra una revision LM como evidencia estructurada."""
    return {"summary": summary}

single_system = toolkit.AgenticSystem(runtime=local_runtime)
policy = toolkit.RunPolicy(max_tool_calls=1, temperature=0.0, trace="compact")
contract = toolkit.AgentContract(must_call=["solve_arithmetic"], tool_expectation=toolkit.expect.exactly("solve_arithmetic"), completion="when_required_tools_satisfied")
agent = single_system.agent(name="single_solver", instructions="Resuelve el problema estructurado.", tools=[solve_arithmetic], engine="python-runtime", runtime=local_runtime, contract=contract, policy=policy)
explainer = single_system.agent(name="single_lm_explainer", instructions="Explica la solucion sin cambiar numeros.", tools=[record_review], runtime=lm_runtime, policy=toolkit.RunPolicy.for_mode("eval"))
toolkit.show({
    "system": single_system.inspect(),
    "agent": agent.info(),
    "optional_explainer": explainer.info(),
}, title="Single AgenticSystem ownership")


## 3) Ejecutar, normalizar y observar

`compose_result` conserva la evidencia del solver y agrega el diagnostico del explainer sin permitir que un fallo opcional contradiga el resultado determinista.


In [ ]:
solve = agent.run({"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}})
explanation = None
explanation_attempt = None
explanation_diagnostic = {
    "status": "skipped",
    "provider": lm_resolution.get("selected_provider"),
    "reason": lm_resolution.get("reason"),
}
if lm_available:
    explanation_attempt = explainer.run(str(solve.data))
    if explanation_attempt.ok:
        explanation = explanation_attempt
        explanation_diagnostic = {
            "status": "ok",
            "provider": explanation_attempt.engine,
            "model": explanation_attempt.model,
        }
    else:
        explanation_diagnostic = {
            "status": "degraded",
            "provider": explanation_attempt.engine,
            "errors": explanation_attempt.errors,
        }
        toolkit.show(explanation_diagnostic, title="Optional LM explanation degraded")
else:
    toolkit.show(explanation_diagnostic, title="Optional LM explanation skipped")
explanation_result = explanation
explanation_text = explanation.text if explanation else None

final = toolkit.final_answer(
    {"procedimiento": solve.data["procedure"], "resultado_final": solve.data["result"], "explicacion_lm": explanation_text},
    schema=toolkit.output_schema(fields=["procedimiento", "resultado_final", "explicacion_lm"]),
)
result = toolkit.compose_result(
    text="Single AgenticSystem end-to-end executed.",
    data=final,
    results=[solve, explanation_result],
    mode="single-agentic-system",
    input=USER_PROMPT,
    meta={"runtime_auto_resolution": lm_resolution, "optional_lm_review": explanation_diagnostic},
)
lineage = result.lineage(name="fundamentals.single_agentic_system", question=USER_PROMPT, goal="Explicar el ciclo completo de un solo sistema.")
toolkit.human_result(result, title="Human result - Single AgenticSystem end-to-end", pretty=PRETTY, show_lineage=True, lineage=lineage)

## 4) Evaluar el mismo Agent

El Environment modela un episodio y `run_eval` valida el agente con un caso repetible. Esta seccion integra las APIs aprendidas en el notebook 10; no redefine su semantica.


In [ ]:
def transition_fn(row: dict, action: dict | None, info: dict) -> dict:
    out = agent.run({"tool": "solve_arithmetic", "input": {"numbers": row["numbers"]}}).data
    return {"result": out["result"], "expected": row["expected"], "ok": out["result"] == row["expected"]}

def reward_fn(state: dict) -> float:
    return 1.0 if state.get("ok") else 0.0

env_records = [{"numbers": NUMBERS, "expected": EXPECTED}]
env = toolkit.AgenticEnvironment(name="single_system_env", records=env_records, initial_memory={}, transition_fn=transition_fn, reward_fn=reward_fn)
env.reset()
_, reward, terminated, truncated, info = env.step()
report = toolkit.run_eval(agent, [{"id": "default", "input": {"tool": "solve_arithmetic", "input": {"numbers": NUMBERS}}, "expected": {"result": EXPECTED}}])
toolkit.show({"env_summary": env.summary(), "step": info["transition"], "eval_report": report.to_dict()})


## Lo que aprendiste

- Un `AgenticSystem` puede ser el owner de Agents con runtimes distintos.
- La ruta obligatoria debe producir un `RunResult` valido sin depender del reviewer LM.
- Environment y Eval reutilizan el mismo Agent y el mismo resultado esperado.
- Una degradacion opcional pertenece a metadata diagnostica, no al estado de exito del calculo.
- Lineage explica que se ejecuto, que evidencia existe y que parte fue opcional.


## Coverage API de este notebook

La siguiente celda enumera la superficie publica materializada por el recorrido end-to-end.


In [ ]:
api_coverage = [
    "AgenticSystem",
    "AgenticSystem.agent",
    "runtime(provider='python-runtime')",
    "runtime(provider='auto')",
    "AgentContract",
    "RunPolicy",
    "final_answer",
    "compose_result",
    "RunResult.lineage",
    "AgenticEnvironment",
    "run_eval",
]
toolkit.show({
    "notebook": "11_single_agentic_system_api.ipynb",
    "api_coverage": api_coverage,
})

## Simbolos API explicados

- `AgenticSystem`: owner del runtime y de los Agents del recorrido.
- `AgenticSystem.agent`: registra Agents con runtime, contrato y policy explicitos.
- `final_answer` / `output_schema`: estabilizan la respuesta de usuario.
- `compose_result`: combina resultados obligatorios y opcionales con trazabilidad.
- `RunResult.lineage`: proyecta evidencia a una explicacion humana.
- `AgenticEnvironment` / `run_eval`: ejecutan y evaluan episodios reproducibles.
